In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import GoogleGenerativeAI, ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import Field, BaseModel
import operator
import os
api_key = os.getenv("GOOGLE_API_KEY")

In [2]:
# Create LLM class
gemini = ChatGoogleGenerativeAI(
    model= "gemini-2.5-pro",
    temperature=1.0,
    max_retries=2,
    google_api_key=api_key,
)

In [3]:
llm1 = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
)

mistral = ChatHuggingFace(llm=llm1)

In [4]:
gemini_flash = ChatGoogleGenerativeAI(
    model= "gemini-1.5-flash",
    temperature=1.0,
    max_retries=2,
    google_api_key=api_key,
)

In [5]:
class ResponseEvalution(BaseModel):
    score: int = Field(description="Score from 1 to 10", ge=0, le=10)
    feedback: str = Field(description="Detailed feedback on the essay")

In [6]:
gemini_with_structured_output = gemini.with_structured_output(ResponseEvalution)

In [7]:
gemini_with_structured_output

RunnableBinding(bound=ChatGoogleGenerativeAI(model='models/gemini-2.5-pro', google_api_key=SecretStr('**********'), temperature=1.0, max_retries=2, client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001D8ECF97730>, default_metadata=()), kwargs={'tools': [{'type': 'function', 'function': {'name': 'ResponseEvalution', 'description': '', 'parameters': {'properties': {'score': {'description': 'Score from 1 to 10', 'maximum': 10, 'minimum': 0, 'type': 'integer'}, 'feedback': {'description': 'Detailed feedback on the essay', 'type': 'string'}}, 'required': ['score', 'feedback'], 'type': 'object'}}}], 'tool_choice': 'ResponseEvalution'}, config={}, config_factories=[])
| PydanticToolsParser(first_tool_only=True, tools=[<class '__main__.ResponseEvalution'>])

In [8]:
essay = """In today's rapidly evolving world, technology plays a pivotal role in shaping our daily lives. From communication to transportation, advancements in technology have revolutionized the way we interact with the world around us. One of the most significant impacts of technology is seen in the field of education. With the advent of online learning platforms and digital resources, students now have access to a wealth of information at their fingertips. This has not only enhanced the learning experience but has also made education more accessible to individuals from diverse backgrounds."""

In [9]:
prompt = f"Please evaluate the following essay and provide a score from 1 to 10 along with detailed feedback:\n\n{essay}"
response = gemini_with_structured_output.invoke(prompt)

In [10]:
response

ResponseEvalution(score=7, feedback="The essay provides a clear and concise introduction to the impact of technology on education. The main idea is well-articulated, and the language is accessible. However, it could be further strengthened by including specific examples of technological tools or platforms to illustrate your points. Additionally, exploring potential counterarguments or challenges related to technology in education would add more depth and critical analysis to the essay. To improve, consider expanding on the points you've made and providing more evidence to support your claims.")

In [12]:
response.score

7

In [13]:
class EssayState(TypedDict):
    eassy: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[list[int], operator.add]
    overall_score: float

In [14]:
graph = StateGraph(EssayState)

In [15]:
def evaluate_language(state: EssayState) -> EssayState:
    prompt = f"Please evaluate the following essay for language quality and provide a score from 1 to 10 along with detailed feedback:\n\n{state['eassy']}"
    output = gemini_with_structured_output.invoke(prompt)
    return {"language_feedback": output.feedback, "individual_scores": [output.score]}

In [16]:
def evaluate_analysis(state: EssayState) -> EssayState:
    prompt = f"Please evaluate the following essay for content analysis and provide a score from 1 to 10 along with detailed feedback:\n\n{state['eassy']}"
    output = gemini_with_structured_output.invoke(prompt)
    return {"analysis_feedback": output.feedback, "individual_scores": [output.score]}

In [17]:
def evaluate_clarity(state: EssayState) -> EssayState:
    prompt = f"Please evaluate the following essay for clarity and coherence and provide a score from 1 to 10 along with detailed feedback:\n\n{state['eassy']}"
    output = gemini_with_structured_output.invoke(prompt)
    return {"clarity_feedback": output.feedback, "individual_scores": [output.score]}

In [18]:
def evaluate_overall(state: EssayState) -> EssayState:
    prompt = f"Based on the previous evaluations feednbacks, please provide an overall assessment of the essay along with detailed feedback: Language Feedback: {state['language_feedback']} \n\nAnalysis Feedback: {state['analysis_feedback']} \n\nClarity Feedback: {state['clarity_feedback']}"
    output = gemini.invoke(prompt).content
    overall_score = sum(state['individual_scores']) / len(state['individual_scores'])
    return {"overall_feedback": output, "overall_score": overall_score}

In [19]:
#add node to the graph
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_clarity", evaluate_clarity)
graph.add_node("evaluate_overall", evaluate_overall)

In [20]:
#add edges to the graph
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_clarity")
graph.add_edge("evaluate_language", "evaluate_overall")
graph.add_edge("evaluate_analysis", "evaluate_overall")
graph.add_edge("evaluate_clarity", "evaluate_overall")
graph.add_edge("evaluate_overall", END)

In [21]:
workflow = graph.compile()

In [22]:
workflow.get_graph().print_ascii()

                                    +-----------+                                    
                                   *| __start__ |*                                   
                              ***** +-----------+ *****                              
                        ******            *            ******                        
                   *****                  *                  *****                   
                ***                       *                       ***                
+-------------------+           +------------------+           +-------------------+ 
| evaluate_analysis |           | evaluate_clarity |           | evaluate_language | 
+-------------------+***        +------------------+         **+-------------------+ 
                        ******            *            ******                        
                              *****       *       *****                              
                                   ***    *    ***    

In [23]:
essay2 = """India is one of the rich contry in world becaus it has many peoples. The economy is mostly based on farming and shoping malls. India produces lot of cars and phones and space rockets are very cheap here. The government prints money very much and that is why there is no problem in prices. The main income is from IT workers who sit at home and earn millions of dollers. India has no poverty becaus everyone have house and food. The stock market is always up and nobody lose money here. The main currency is the rupees but it is same value like dollar now. Exporting software and mangoes makes India very powerful in world trade. Also India is biggest oil producer because everyone drive car and planes use petrol from India. Inflation is small and bank always give high interest rate like 50%. People save money in piggy banks mostly and government dont tax much. Foreign investors dont like India much but it is ok because India is self suficient. So India economy is the best in world and everyone should invest here without thinking."""

In [24]:
initial_state = {"eassy": essay2}

In [25]:
final_state = workflow.invoke(initial_state)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 2
Please retry in 46.62006694s. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 2
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 46
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 2
Please retry in 44.157479439s. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 2
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 44
}
]

In [ ]:
final_state

{'eassy': 'India is one of the rich contry in world becaus it has many peoples. The economy is mostly based on farming and shoping malls. India produces lot of cars and phones and space rockets are very cheap here. The government prints money very much and that is why there is no problem in prices. The main income is from IT workers who sit at home and earn millions of dollers. India has no poverty becaus everyone have house and food. The stock market is always up and nobody lose money here. The main currency is the rupees but it is same value like dollar now. Exporting software and mangoes makes India very powerful in world trade. Also India is biggest oil producer because everyone drive car and planes use petrol from India. Inflation is small and bank always give high interest rate like 50%. People save money in piggy banks mostly and government dont tax much. Foreign investors dont like India much but it is ok because India is self suficient. So India economy is the best in world an